# 🔬 Exploratory Data Analysis (EDA)
## Twitter Sentiment Analysis — Mental Health Topic

**Isi notebook ini:**
- Section 5: Exploratory Data Analysis
- Section 6: Visualisasi Data (6 chart)

> **Prasyarat:** Jalankan `01_data_wrangling.ipynb` terlebih dahulu.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import ast
import json
import re
import warnings
from pathlib import Path

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

In [2]:
# ── Konfigurasi ──────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../datasets/processed')
OUTPUT_DIR    = Path('../reports/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Palet warna sentimen
PAL  = {'Positive': '#00C896', 'Neutral': '#4A9EFF', 'Negative': '#FF5A5A'}

# Tema dark untuk matplotlib
THEME = {
    'BG'  : '#0D1117',
    'CARD': '#161B22',
    'GRID': '#21262D',
    'TXT' : '#E6EDF3',
    'SUB' : '#8B949E',
    'YEL' : '#F0C040',
}

plt.rcParams.update({
    'figure.facecolor' : THEME['BG'],
    'axes.facecolor'   : THEME['CARD'],
    'axes.edgecolor'   : THEME['GRID'],
    'text.color'       : THEME['TXT'],
    'xtick.color'      : THEME['SUB'],
    'ytick.color'      : THEME['SUB'],
    'grid.color'       : THEME['GRID'],
    'axes.labelcolor'  : THEME['TXT'],
    'axes.titlecolor'  : THEME['TXT'],
    'legend.facecolor' : THEME['CARD'],
    'legend.edgecolor' : THEME['GRID'],
    'legend.labelcolor': THEME['TXT'],
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.titlepad'    : 10,
})

In [3]:
# ── Helper Functions ─────────────────────────────────────────────────────────
def save_figure(fig: plt.Figure, name: str) -> None:
    """Simpan figure ke OUTPUT_DIR dengan format PNG."""
    path = OUTPUT_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=THEME['BG'])
    plt.close(fig)
    print(f'✅ Disimpan → {path}')


def parse_hashtags(raw: str) -> list[str]:
    """Re-parse hashtag dari kolom hashtags jika hashtag_list belum ada."""
    try:
        items = ast.literal_eval(str(raw))
        return [
            re.sub(r"['\"\s#]", '', str(item)).lower().strip()
            for item in items
            if len(str(item).strip()) > 1
        ]
    except Exception:
        return []

In [4]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(PROCESSED_DIR / 'twitter_clean.csv')

with open(PROCESSED_DIR / 'top_hashtags.json') as f:
    top_hashtags = json.load(f)

# Re-parse hashtag_list jika kolom belum tersedia
if 'hashtag_list' not in df.columns:
    df['hashtag_list'] = df['hashtags'].apply(parse_hashtags)

# Sub-grup per sentimen (digunakan berulang di seluruh notebook)
pos = df[df['sentiment'] == 'Positive']
neu = df[df['sentiment'] == 'Neutral']
neg = df[df['sentiment'] == 'Negative']

print(f'✅ Dataset dimuat: {len(df):,} baris × {len(df.columns)} kolom')
print(f'   Positive: {len(pos):,} | Neutral: {len(neu):,} | Negative: {len(neg):,}')

✅ Dataset dimuat: 8,439 baris × 9 kolom
   Positive: 1,505 | Neutral: 2,804 | Negative: 4,130


---
## 5. Exploratory Data Analysis

In [5]:
# 5.1 Distribusi Sentimen (Pasca Cleaning)
dist = df['sentiment'].value_counts()
for sentiment, count in dist.items():
    bar = '█' * int(count / 50)
    print(f'  {sentiment:<10} {count:>5,}  ({count / len(df) * 100:.1f}%)  {bar}')

  Negative   4,130  (48.9%)  ██████████████████████████████████████████████████████████████████████████████████
  Neutral    2,804  (33.2%)  ████████████████████████████████████████████████████████
  Positive   1,505  (17.8%)  ██████████████████████████████


In [6]:
# 5.2 Statistik Deskriptif per Sentimen
feature_cols = ['text_length', 'hashtag_count']

df.groupby('sentiment')[feature_cols].agg(['mean', 'median', 'std']).round(3)


text_length                hashtag_count              
                 mean median     std          mean median    std
sentiment                                                       
Negative      182.048  194.0  77.175         5.247    3.0  5.957
Neutral       179.208  191.0  72.841         3.740    3.0  4.136
Positive      180.033  191.0  72.834         3.728    3.0  4.153

> **Catatan:** Analisis fitur boolean dihapus karena merupakan hasil feature engineering.


In [7]:
# 5.4 Korelasi Fitur Numerik dengan Target ('labels')
numeric_cols = ['text_length', 'hashtag_count', 'labels']
corr = df[numeric_cols].corr()['labels'].drop('labels').sort_values(ascending=False)

print('Korelasi fitur dengan target (labels):')
for feat, val in corr.items():
    bar = '+' * int(abs(val) * 20) if val > 0 else '-' * int(abs(val) * 20)
    print(f'  {feat:<22} {val:+.4f}  {bar}')


Korelasi fitur dengan target (labels):
  text_length            -0.0135  
  hashtag_count          -0.1309  --


In [8]:
# 5.5 Top 5 Hashtag per Sentimen
for sentiment in ['Positive', 'Neutral', 'Negative']:
    print(f'\nSentimen {sentiment}:')
    for tag, count in top_hashtags[sentiment][:5]:
        bar = '█' * int(count / 30)
        print(f'  #{tag:<30} {count:>5,}  {bar}')


Sentimen Positive:
  #mentalhealth                     415  █████████████
  #stress                           331  ███████████
  #mentalhealthmatters              202  ██████
  #tired                            173  █████
  #anxiety                          110  ███

Sentimen Neutral:
  #mentalhealth                     771  █████████████████████████
  #stress                           652  █████████████████████
  #mentalhealthmatters              406  █████████████
  #tired                            295  █████████
  #anxiety                          210  ███████

Sentimen Negative:
  #happy                          1,012  █████████████████████████████████
  #excited                          757  █████████████████████████
  #mentalhealth                     558  ██████████████████
  #stress                           547  ██████████████████
  #love                             378  ████████████


---
## 6. Visualisasi Data

In [9]:
# Viz 1 — EDA Overview Dashboard (1×2 grid)
BG   = THEME['BG']
CARD = THEME['CARD']
GRID = THEME['GRID']
TXT  = THEME['TXT']
SUB  = THEME['SUB']

fig = plt.figure(figsize=(14, 6), facecolor=BG)
fig.suptitle(
    'Twitter Sentiment Analysis — EDA Overview',
    fontsize=22, fontweight='bold', color=TXT, y=1.01,
)
gs = gridspec.GridSpec(1, 2, figure=fig, hspace=0.45, wspace=0.38)

# Panel A: Distribusi Sentimen (BQ1)
ax_a = fig.add_subplot(gs[0, 0])
counts = df['sentiment'].value_counts().reindex(['Positive', 'Neutral', 'Negative'])
bars = ax_a.bar(
    counts.index, counts.values,
    color=[PAL[s] for s in counts.index],
    width=0.52, edgecolor=BG, linewidth=1.4,
)
for bar in bars:
    h = bar.get_height()
    ax_a.text(
        bar.get_x() + bar.get_width() / 2, h + 40,
        f'{int(h):,}\n({h / len(df) * 100:.1f}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold', color=TXT,
    )
ax_a.set_title('BQ1 — Distribusi Sentimen')
ax_a.set_ylabel('Jumlah Tweet')
ax_a.set_ylim(0, counts.max() * 1.28)
ax_a.grid(axis='y', alpha=0.2)

# Panel B: Hashtag Count Violin (BQ3)
ax_d = fig.add_subplot(gs[0, 1])
data_vl = [df[df['sentiment'] == s]['hashtag_count'].values for s in ['Positive', 'Neutral', 'Negative']]
parts = ax_d.violinplot(data_vl, positions=[1, 2, 3], showmedians=True, showextrema=False)
for body, key in zip(parts['bodies'], ['Positive', 'Neutral', 'Negative']):
    body.set_facecolor(PAL[key])
    body.set_alpha(0.75)
    body.set_edgecolor(BG)
parts['cmedians'].set_color('white')
parts['cmedians'].set_linewidth(2)
ax_d.set_xticks([1, 2, 3])
ax_d.set_xticklabels(['Positive', 'Neutral', 'Negative'])
ax_d.set_title('BQ3 — Hashtag Count per Sentimen')
ax_d.set_ylabel('Jumlah Hashtag')
ax_d.grid(axis='y', alpha=0.2)
for sentiment, x_pos in zip(['Positive', 'Neutral', 'Negative'], [1, 2, 3]):
    mean_val = df[df['sentiment'] == sentiment]['hashtag_count'].mean()
    ax_d.text(x_pos, ax_d.get_ylim()[1] * 0.95, f'μ={mean_val:.1f}',
              ha='center', va='top', fontsize=9, color=PAL[sentiment], fontweight='bold')

save_figure(fig, 'viz1_eda_overview')


✅ Disimpan → ..\reports\figures\viz1_eda_overview.png


In [10]:
# Viz 2 — Top 12 Hashtag per Sentimen (BQ2 & BQ5)
fig2, axes2 = plt.subplots(1, 3, figsize=(21, 7), facecolor=BG)
fig2.suptitle(
    'BQ2 & BQ5 — Top 12 Hashtag per Kategori Sentimen',
    fontsize=18, fontweight='bold', color=TXT, y=1.02,
)

cmaps = {'Positive': plt.cm.Greens, 'Neutral': plt.cm.Blues, 'Negative': plt.cm.Reds}

for ax, sentiment in zip(axes2, ['Positive', 'Neutral', 'Negative']):
    ax.set_facecolor(CARD)
    data = top_hashtags[sentiment][:12][::-1]
    tags  = [f'#{d[0]}' for d in data]
    freqs = [d[1] for d in data]
    n = len(freqs)
    colors = [cmaps[sentiment](0.35 + 0.55 * (i / max(n - 1, 1))) for i in range(n)]

    bars = ax.barh(tags, freqs, color=colors, edgecolor=BG, height=0.68)
    for bar in bars:
        ax.text(
            bar.get_width() + max(freqs) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{int(bar.get_width()):,}',
            va='center', fontsize=8.5, color=TXT,
        )

    title = f'Sentimen: {sentiment}'
    if sentiment == 'Negative':
        title += '\n(Perhatikan #happy & #excited — ironi!)'
    ax.set_title(title, fontsize=13, fontweight='bold', color=PAL[sentiment], pad=10)
    ax.set_xlabel('Frekuensi', color=SUB)
    ax.tick_params(axis='y', labelcolor=TXT, labelsize=9)
    ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
save_figure(fig2, 'viz2_top_hashtags')

✅ Disimpan → ..\reports\figures\viz2_top_hashtags.png


In [11]:
# Viz 3 — Heatmap Korelasi Antar Fitur
numeric_cols = ['text_length', 'hashtag_count', 'labels']
short_labels  = ['txt_len', 'htag_cnt', 'label']

corr_matrix = df[numeric_cols].corr().round(3)

fig3, ax3 = plt.subplots(figsize=(6, 5), facecolor=BG)
ax3.set_facecolor(CARD)

im = ax3.imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax3.set_xticks(range(len(numeric_cols)))
ax3.set_yticks(range(len(numeric_cols)))
ax3.set_xticklabels(short_labels, rotation=40, ha='right', fontsize=9, color=TXT)
ax3.set_yticklabels(short_labels, fontsize=9, color=TXT)

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        v = corr_matrix.values[i, j]
        ax3.text(
            j, i, f'{v:.2f}',
            ha='center', va='center', fontsize=8, fontweight='bold',
            color='black' if abs(v) < 0.5 else 'white',
        )

plt.colorbar(im, ax=ax3, fraction=0.04, pad=0.02)
ax3.set_title('Heatmap Korelasi Antar Fitur Numerik', fontsize=15, fontweight='bold', color=TXT, pad=12)
plt.tight_layout()
save_figure(fig3, 'viz3_correlation_heatmap')


✅ Disimpan → ..\reports\figures\viz3_correlation_heatmap.png
